# 13장. ONNX 만들기

In [1]:
# [준비 셀] 모든 노트북의 첫 코드 셀입니다. 항상 먼저 실행하세요.
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parents[1]          # docs/manual → 프로젝트 루트
assert (ROOT / "src/bearing_kd").exists(), "docs/manual 폴더에서 노트북을 열어야 합니다"
CODE = ROOT / "docs/manual/code"                 # 챕터별 완성 스크립트 위치
sys.path.insert(0, str(CODE))                    # 스크립트의 함수를 import 해서 쓰기 위해
print("프로젝트 루트 확인 완료:", ROOT.name)

프로젝트 루트 확인 완료: vib-onnx


지금까지 만든 모델은 **PyTorch + MOMENT 패키지**가 있어야 돌아갑니다. 라즈베리파이에 이 무거운 패키지들을 깔기는 어렵습니다. 그래서 모델을 **ONNX**로 변환합니다.

## 13.1 ONNX란?

**ONNX(Open Neural Network Exchange)**는 딥러닝 모델을 저장하는 **공통 파일 형식**입니다.

| | PyTorch 체크포인트 (.pt) | ONNX (.onnx) |
|---|---|---|
| 들어 있는 것 | 숫자(가중치)만 | **계산 순서(그래프) + 숫자** |
| 실행하려면 | 모델을 만드는 파이썬 코드(MOMENT)가 필요 | `onnxruntime`만 있으면 됨 |
| 용도 | 학습, 연구 | 배포 (라즈베리파이, 서버, 모바일) |

`.pt`는 "재료"만 있고 "요리법"은 파이썬 코드에 있습니다. ONNX는 **요리법까지 파일 안에** 들어 있어서 어디서든 같은 요리를 만들 수 있습니다.

## 13.2 변환 방식: 예시 입력을 흘려 보내 계산을 녹화

`torch.onnx.export`는 모델에 **예시 입력**을 한 번 넣어 보면서, 어떤 연산이 어떤 순서로 일어나는지 **녹화**해서 그래프로 저장합니다. 그래서:
- 입력 **모양**이 고정됩니다. 이 모델은 `x_enc [1, 1, 512]`, `input_mask [1, 512]`로 **청크 1개씩**만 받습니다.
- 윈도우 하나를 판정하려면 ONNX를 **8번** 실행하고 평균을 냅니다(15장).

In [2]:
import os, warnings, logging
os.environ["HF_HUB_OFFLINE"] = "1"
warnings.filterwarnings("ignore")
logging.getLogger("torch.onnx").setLevel(logging.ERROR)
import numpy as np, torch

## 13.3 변환 전에 손볼 것 세 가지

### ① 출력을 텐서로: wrapper

MOMENT는 결과를 `logits`, `embeddings` 등이 든 **객체**로 돌려줍니다. ONNX 출력은 텐서여야 하므로 `logits`만 꺼내 주는 얇은 껍데기로 감쌉니다.

```python
class ChunkWrapper(torch.nn.Module):
    def forward(self, x_enc, input_mask):
        return self.model(x_enc=x_enc, input_mask=input_mask).logits
```

### ② RevIN의 `nanmean` 문제

MOMENT의 RevIN은 mask=0인 칸을 NaN으로 바꾼 뒤 `torch.nanmean`으로 평균을 냅니다. 그런데 **ONNX에는 nanmean 연산이 없습니다.** 옛 방식(TorchScript) exporter로 원본을 변환하면 실제로 이렇게 실패합니다.

In [3]:
# 옛 방식 exporter는 실패하면서 그래프 전체(수천 줄)를 쏟아내므로, 별도 프로세스에서 돌리고 에러 줄만 본다
import subprocess, sys
test_code = """
import sys, os, warnings; warnings.filterwarnings('ignore'); os.environ['HF_HUB_OFFLINE'] = '1'
sys.path.insert(0, 'code')
import torch
from ch10_checkpoint import load_full, ROOT
from ch13_export_onnx import ChunkWrapper
m = load_full(ROOT / 'models/MOMENT-1-small', ROOT / 'outputs/manual/student/student_kd_best.pt', teacher=False)
try:
    torch.onnx.export(ChunkWrapper(m).eval(), (torch.randn(1, 1, 512), torch.ones(1, 512)),
                      str(ROOT / 'outputs/manual/onnx/_legacy_test.onnx'), opset_version=18, dynamo=False)
    print('RESULT: 변환 성공')
except Exception as e:
    print('RESULT:', type(e).__name__, '->', str(e).splitlines()[0])
"""
run = subprocess.run([sys.executable, "-c", test_code], capture_output=True, text=True)
print([line for line in run.stdout.splitlines() if line.startswith("RESULT")][0])

RESULT: UnsupportedOperatorError -> Exporting the operator 'aten::nanmean' to ONNX opset version 18 is not supported


그래서 이 프로젝트는 RevIN을 **"유효한 칸만 더해서 개수로 나누기"**로 다시 구현했습니다. 계산 결과는 같고, 기본 연산(ReduceSum, Where, Div)만 씁니다.

```python
valid = (mask > 0) & isfinite(x)
mean  = sum(where(valid, x, 0)) / count(valid)
var   = sum(where(valid, x - mean, 0)²) / count(valid)
out   = (x - mean) / (sqrt(var) + eps)
```

> 참고: 최신 **dynamo 방식** exporter(`dynamo=True`, PyTorch 2.x)는 nanmean을 기본 연산으로 쪼개서 원본도 변환할 수 있습니다. 그래도 교체 구현을 쓰면 exporter 버전과 상관없이 결과가 확실해서 이 프로젝트는 계속 사용합니다.

### ③ head: 평가 모드 동작만

원본 head는 `평균 → dropout → 선형층`입니다. dropout은 학습 중에만 쓰이므로 빼고, NaN이 섞여도 안전한 평균으로 바꿉니다. 학습된 선형층 가중치는 그대로 가져옵니다.

In [4]:
import ch13_export_onnx as ch13
original, wrapper = ch13.build_exportable(ROOT / "outputs/manual/student/student_kd_best.pt")
x, mask = torch.randn(1, 1, 512), torch.ones(1, 512)
mask[:, 416:] = 0
with torch.no_grad():
    a = original(x_enc=x, input_mask=mask).logits
    b = wrapper(x, mask)
print("원본", a.numpy().round(5), " 교체본", b.numpy().round(5), " 차이", float((a - b).abs().max()))

Loading weights from local directory
원본 [[ 0.61987 -0.45662]]  교체본 [[ 0.61987 -0.45662]]  차이 0.0


## 13.4 변환하기

```python
torch.onnx.export(
    wrapper,                                   # 변환할 모델
    (dummy_x, dummy_mask),                     # 예시 입력 (모양만 중요)
    "student.onnx",
    input_names=["x_enc", "input_mask"],       # ONNX 입력 이름
    output_names=["logits"],                   # ONNX 출력 이름
    opset_version=18,                          # 사용할 ONNX 연산 버전
    dynamo=True,                               # 새 방식 exporter
    external_data=True,                        # 가중치를 .onnx.data 파일로 분리
)
```

- **opset**: ONNX 연산 목록의 버전입니다. 숫자가 클수록 새 연산을 쓸 수 있지만, 실행하는 쪽(onnxruntime)도 그 버전을 지원해야 합니다.
- **external_data**: ONNX 파일 하나는 protobuf 형식 제한 때문에 2 GB를 넘을 수 없습니다. 가중치를 별도 파일로 빼 두면 구조 파일이 작아지고 다루기 쉽습니다. **대신 두 파일을 항상 같이** 옮겨야 합니다.

## 13.5 검증: 파일이 만들어졌다고 끝이 아니다

변환은 "녹화"라서, 녹화 중에 조건문이 한쪽으로만 기록되는 등 **조용히 다른 모델**이 될 수 있습니다. 그래서 여러 입력을 넣어 **PyTorch 원본과 ONNX Runtime 결과를 직접 비교**합니다.

| 입력 | 확인하려는 것 |
|---|---|
| 무작위 | 일반적인 경우 |
| 전부 0, 상수 | 표준편차 0일 때 나눗셈 (eps 처리) |
| 마지막 96칸 패딩 | mask=0 처리 |
| 실제 윈도우의 청크 8개 | 실제 데이터 |

허용 오차는 `rtol=1e-4, atol=1e-5`입니다. 컴퓨터 계산은 순서만 달라도 마지막 자리가 조금씩 다르므로, 완전히 같을 것을 요구하지 않습니다.

In [5]:
%run code/ch13_export_onnx.py --overwrite

Loading weights from local directory



입력                         PyTorch 원본           ONNX Runtime  차이
random             [+1.4875, -1.3533]     [+1.4875, -1.3533]  3.6e-07
zeros              [+3.1138, -2.8347]     [+3.1138, -2.8348]  1.7e-06
constant           [+3.1138, -2.8347]     [+3.1138, -2.8348]  1.7e-06
padded             [+1.6614, -1.4998]     [+1.6614, -1.4998]  1.8e-06
real chunk 1       [+1.7915, -1.6877]     [+1.7915, -1.6877]  8.3e-07
real chunk 2       [+1.1966, -1.1001]     [+1.1966, -1.1001]  9.5e-07
real chunk 3       [+1.7075, -1.6171]     [+1.7075, -1.6171]  1.9e-06
real chunk 4       [+1.3472, -1.1950]     [+1.3472, -1.1950]  6.0e-07
real chunk 5       [+1.8867, -1.8099]     [+1.8867, -1.8099]  4.8e-07
real chunk 6       [+1.3846, -1.2400]     [+1.3846, -1.2400]  7.2e-07
real chunk 7       [+1.6604, -1.6093]     [+1.6604, -1.6093]  1.2e-06
real chunk 8       [+1.6617, -1.5105]     [+1.6617, -1.5105]  1.1e-06

저장: outputs/manual/onnx/student.onnx (0.9 MB) + student.onnx.data (86.1 MB)
검증 12개 입력 통과, 최대 

In [6]:
import json
print(json.dumps(json.loads((ROOT / "outputs/manual/onnx/student.verification.json").read_text()), indent=2))
(ROOT / "outputs/manual/onnx/_legacy_test.onnx").unlink(missing_ok=True)

{
  "checkpoint": "outputs/manual/student/student_kd_best.pt",
  "onnx": "outputs/manual/onnx/student.onnx",
  "cases": 12,
  "max_absolute_error": 1.9073486328125e-06
}


## 13.6 윈도우 전체로 한 번 더 확인

청크 단위뿐 아니라 **윈도우 판정**(8청크 평균)도 PyTorch와 같은지 test 윈도우 전체로 비교합니다.

In [7]:
import onnxruntime as ort
from ch06_dataset import WindowDataset
from ch12_evaluate import predict
from ch15_pi_deploy import predict_window

split = json.loads((ROOT / "outputs/manual/split_holdout.json").read_text())
test = WindowDataset([ROOT / f for f in split["test"]])
torch_logits = predict(original, test, "cpu")
sess = ort.InferenceSession(str(ROOT / "outputs/manual/onnx/student.onnx"), providers=["CPUExecutionProvider"])
onnx_logits = np.array([predict_window(sess, w) for w in test.windows])
print(f"윈도우 {len(test)}개: 최대 차이 {np.abs(torch_logits - onnx_logits).max():.1e}, "
      f"예측 일치 {(torch_logits.argmax(1) == onnx_logits.argmax(1)).sum()}/{len(test)}")

윈도우 113개: 최대 차이 6.0e-07, 예측 일치 113/113


실제 프로젝트 명령:
```bash
bearing-kd export --checkpoint outputs/runs/v3/student/student_kd_best.pt --output outputs/runs/v3/onnx/student.onnx \
    --sample data/processed/windows_v3/KB24/N09_M07_F10_KB24_1_windows.npy
```

## 13장 완성 코드: `code/ch13_export_onnx.py`

In [8]:
from IPython.display import Code
Code(filename=str(CODE / "ch13_export_onnx.py"), language="python")

"""13장 완성 코드: Student 체크포인트를 ONNX로 변환하고, PyTorch와 결과가 같은지 검증한다.

src/bearing_kd/export.py를 읽기 쉽게 줄인 버전이다.
    1) 체크포인트 불러오기 (MOMENT Small + 학습된 head)
    2) RevIN, head를 ONNX 친화적인 구현으로 교체 (결과는 같음)
    3) logits만 돌려주는 wrapper로 감싸기
    4) torch.onnx.export (dynamo 방식, opset 18, 가중치는 .onnx.data로 분리)
    5) onnx.checker로 형식 검사 + onnxruntime 결과를 PyTorch와 비교

실행 (프로젝트 루트에서, 11장 결과 사용):
    python docs/manual/code/ch13_export_onnx.py --overwrite
    python docs/manual/code/ch13_export_onnx.py --checkpoint outputs/runs/v3/student/student_kd_best.pt \
        --output outputs/manual/onnx_v3/student.onnx --overwrite
"""

import argparse
import copy
import json
import logging
import warnings
from pathlib import Path

import numpy as np
import torch

from ch07_moment import ROOT
from ch10_checkpoint import load_full

CHUNK = 512
logging.getLogger("torch.onnx").setLevel(logging.ERROR)   # "torchvision 없음" 같은 무관한 경고 숨김


class ExportableRevIN(torch.nn.Module):
    """MOMENT RevIN과 같은 계산을 nanmean 없이 한다.

    원본은 mask=0인 칸을 NaN으로 바꾼 뒤 nanmean/nanstd를 쓴다. ONNX에는 nanmean 연산이 없어서
    옛 방식(TorchScript) exporter는 변환에 실패한다. 여기서는 "유효한 칸만 더하고 개수로 나누기"로 바꾼다.
    """

    def __init__(self, original):
        super().__init__()
        self.eps = original.eps

    def forward(self, x, mode="norm", mask=None):
        mask = torch.ones(x.shape[0], x.shape[-1], dtype=x.dtype) if mask is None else mask
        valid = (mask.unsqueeze(1).expand_as(x) > 0) & torch.isfinite(x)       # 유효한 칸
        count = valid.sum(dim=-1, keepdim=True).clamp_min(1).to(x.dtype)
        mean = torch.where(valid, x, torch.zeros_like(x)).sum(dim=-1, keepdim=True) / count
        var = torch.where(valid, x - mean, torch.zeros_like(x)).square().sum(dim=-1, keepdim=True) / count
        return (x - mean) / (torch.sqrt(var) + self.eps)


class ExportableHead(torch.nn.Module):
    """분류 head: 패치 특징을 평균 → 선형층. 평가 모드에서는 dropout이 꺼지므로 빼고, NaN이 섞여도 안전한 평균을 쓴다."""

    def __init__(self, original):
        super().__init__()
        self.linear = original.linear          # 학습된 가중치를 그대로 사용

    def forward(self, x, input_mask=None):
        valid = ~torch.isnan(x)
        pooled = torch.where(valid, x, torch.zeros_like(x)).sum(dim=1) / valid.sum(dim=1).clamp_min(1).to(x.dtype)
        return self.linear(pooled)


class ChunkWrapper(torch.nn.Module):
    """MOMENT는 결과를 여러 필드가 든 객체로 돌려준다. ONNX 출력은 텐서여야 하므로 logits만 꺼낸다."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x_enc, input_mask):
        return self.model(x_enc=x_enc, input_mask=input_mask).logits


def build_exportable(checkpoint):
    original = load_full(ROOT / "models/MOMENT-1-small", checkpoint, teacher=False)
    model = copy.deepcopy(original)
    model.normalizer = ExportableRevIN(model.normalizer)
    model.head = ExportableHead(model.head)
    return original, ChunkWrapper(model).eval()


def parity_cases(sample_window=None):
    """변환이 맞는지 볼 입력들: 무작위, 0, 상수, 마지막 96칸 패딩, 실제 윈도우의 청크 8개."""
    g = torch.Generator().manual_seed(42)
    x = torch.randn(1, 1, CHUNK, generator=g)
    ones = torch.ones(1, CHUNK)
    padded_x, padded_mask = x.clone(), ones.clone()
    padded_x[..., 416:], padded_mask[:, 416:] = 0, 0
    cases = [("random", x, ones), ("zeros", torch.zeros_like(x), ones), ("constant", torch.ones_like(x), ones),
             ("padded", padded_x, padded_mask)]
    if sample_window is not None:
        w = np.pad(np.asarray(sample_window, dtype=np.float32), (0, 96)).reshape(8, 1, CHUNK)
        m = (np.arange(8 * CHUNK) < 4000).astype(np.float32).reshape(8, CHUNK)
        cases += [(f"real chunk {i + 1}", torch.from_numpy(w[i:i + 1]), torch.from_numpy(m[i:i + 1])) for i in range(8)]
    return cases


def export(checkpoint, output, sample_window=None):
    import onnx
    import onnxruntime as ort

    original, wrapper = build_exportable(checkpoint)
    output.parent.mkdir(parents=True, exist_ok=True)
    dummy = (torch.randn(1, 1, CHUNK), torch.ones(1, C